# 03 · Retrieval: the same question, ranked by each method
Inspects `ragtraining.retrieve.search` (Phase 3). For a few questions, the top-5 chunks per method are shown side by side.
Everything runs locally (Ollama + local reranker): **$0**. Needs `ollama serve` running.

In [ ]:
import sys, json, pickle; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', 70); pd.set_option('display.width', 200)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
from ragtraining.retrieve.search import Retriever
r = Retriever()
METHODS = {'cosine (Qwen3)': dict(method='cosine'), 'euclid (nomic raw)': dict(method='euclid'),
           'BM25': dict(method='bm25'), 'hybrid RRF': dict(method='hybrid', how='rrf'),
           'hybrid + local rerank': dict(method='hybrid', how='rrf', rerank='local')}

def label(x):
    page = f'p.{int(x.page_start)}' if pd.notna(x.page_start) else (x.section if isinstance(x.section, str) else '')[:15]
    return f"{x.course[:14]} · {x.path.split('/')[-1][:28]} · {page}"

def side_by_side(q, k=5):
    res = {n: r.search(q, k=k, **cfg) for n, cfg in METHODS.items()}
    print('Q:', q)
    return pd.DataFrame({n: [label(x) for x in df.itertuples()] for n, df in res.items()}, index=range(1, k + 1)), res

## 1. A normal exam question
All methods agree on the course. They differ on **which pages** rise to the top.

In [ ]:
t1, res1 = side_by_side('How does two-phase locking guarantee that a schedule is serializable?'); t1

## 2. A paraphrase with none of the slide's keywords → dense wins
The question describes the Feistel idea without saying "Feistel", "round" or "XOR". BM25 has nothing to match.

In [ ]:
t2, _ = side_by_side("Why can a block cipher be decrypted even when its inner scrambling function can't be inverted?"); t2

## 3. An exact technical term → BM25 shines
Rare names and formulas ("Sharpe ratio") are exactly what keyword search is built for.

In [ ]:
t3, _ = side_by_side('Sharpe ratio formula'); t3

## 4. How RRF fuses two rankings
**score = 1/(60 + rank_dense) + 1/(60 + rank_bm25)**. It uses only ranks, so the scales of the two scores never need
calibrating. A chunk ranked well by *both* beats one ranked #1 by only one.

In [ ]:
q = 'How does two-phase locking guarantee that a schedule is serializable?'
rank = lambda s: pd.Series(s).rank(ascending=False, method='first').astype(int).to_numpy()
rd, rb = rank(r.cosine(q)), rank(r.bm25_scores(q))
top = r.search(q, k=8, method='hybrid', how='rrf', use_type_weights=False)
pos = {c: i for i, c in enumerate(r.chunks.chunk_id)}
idx = [pos[c] for c in top.chunk_id]
pd.DataFrame({'chunk': [label(x) for x in top.itertuples()], 'dense rank': rd[idx], 'BM25 rank': rb[idx],
              'RRF score': (1 / (60 + rd[idx]) + 1 / (60 + rb[idx])).round(4)}, index=range(1, 9))

## 5. How different are the methods? (overlap of top-10)
**Jaccard** = shared chunks / all chunks in the two top-10 lists. Low overlap between dense and BM25 is *why* hybrid
fusion helps: they find different relevant pages.

In [ ]:
tops = {n: set(r.search(q, k=10, **cfg).chunk_id) for n, cfg in METHODS.items()}
names = list(tops)
J = pd.DataFrame([[len(tops[a] & tops[b]) / len(tops[a] | tops[b]) for b in names] for a in names], index=names, columns=names)
fig, ax = plt.subplots(figsize=(6.5, 5)); im = ax.imshow(J, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(names)), names, rotation=40, ha='right'); ax.set_yticks(range(len(names)), names)
for a in range(len(names)):
    for b in range(len(names)): ax.text(b, a, f'{J.iat[a, b]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title('top-10 overlap (Jaccard), 2PL question'); plt.tight_layout(); plt.show()